**00. Creación del dataset**

## **Proyecto final de 4Geeks: Sistema de alerta de seguridad urbana**
- **Equipo:** Alessandra | Natalia | Andrés  
- **Objetivo de este notebook:** explicar de dónde vienen los datos, cuáles datasets combinamos, cómo los unificamos y qué significa cada columna del dataset final.

### **1. Planteamiento del problema**
**Contexto**

Los espacios públicos en las ciudades, como las calles, las plazas, el transporte público y los aparcamientos, son lugares donde suenan muchos ruidos diferentes. En estos lugares, los ruidos cotidianos conviven con situaciones de peligro real. Ser capaz de distinguir automáticamente entre un perro ladrando, niños jugando y un disparo tiene aplicaciones directas en:
- **Seguridad municipal:** detectar disparos o accidentes de manera automática.
- **Accesibilidad y smartwatches de asistencia:** alertas visuales o de vibración para personas con discapacidades auditivas.
- **Sistemas de seguridad inteligentes:** integrar audio con cámaras de seguridad municipales.
- **Redes de vigilancia municipal y aplicaciones de seguridad ciudadana:** notificar a los usuarios en zonas peligrosas en tiempo real.
- **Smart cities:** crear un mapa de contaminación acústica en tiempo real para distintas zonas.
- **Seguros y análisis forense:** registrar y clasificar incidentes de manera automática.

Nuestro proyecto propone entrenar un modelo de clasificación de audio que pueda identificar en tiempo real sonidos urbanos peligrosos o relevantes, con el objetivo de enviar alertas automáticas en espacios públicos. 

Dado un fragmento de audio de corta duración grabado en un entorno urbano, nuestro modelo podrá **clasificar qué tipo de sonido es** y determinar **si representa una situación de emergencia o peligro**.

Este es un problema de **clasificación multiclase supervisada** basado en señales de audio. La variable objetivo principal es la clase del sonido (`human_label`).

### **2. Fuentes de datos**
Para alcanzar el mínimo de **60.000 filas** y **20 columnas** requerido, combinamos varios datasets públicos complementarios. Cada uno aporta clases y contextos sonoros distintos que enriquecen el modelo final.

### **2.1 UrbanSound8K - Kaggle**
**Fuente:** https://www.kaggle.com/datasets/chrisfilo/urbansound8k

Este dataset contiene **8.732 fragmentos** de audio de hasta 4 segundos, etiquetados y extraídos de grabaciones reales subidas a la plataforma Freesound. Cada fragmento pertenece a una de 10 clases de sonidos urbanos y tiene un archivo de metadatos llamado UrbanSound8K.csv. Es el dataset más citado en la literatura de clasificación de audio urbano.

| Clase | Relevancia para el sistema de alertas |
|---|---|
| `gun_shot` | crítica |
| `siren` | crítica |
| `car_horn` | alta |
| `dog_bark` | media |
| `drilling` | media |
| `jackhammer` | media |
| `engine_idling` | baja |
| `children_playing` | baja |
| `street_music` | baja |
| `air_conditioner` | baja |

Los autores del dataset advierten que no debemos reordenar los datos. El dataset viene ya dividido en 10 folds y los fragmentos del mismo archivo original siempre están en el mismo fold, evitando data leakage. Seguimos la **validación cruzada de 10 folds** con las divisiones predefinidas.

### **2.2 FSD50K (Zenodo)**
**Fuente:** https://zenodo.org/records/4060432

Este dataset contiene **51.197 fragmentos** de audio de Freesound distribuidos de forma desigual en 200 clases extraídas de la ontología AudioSet. FSD50K fue creado en el Grupo de Tecnología Musical (MTG) de la Universitat Pompeu Fabra.

### **2.3 ESC-50 (GitHub)**
**Fuente:** https://github.com/karolpiczak/ESC-50  

El dataset ESC-50 es una colección etiquetada de **2.000 grabaciones de audio ambiental**, ideal para evaluar métodos de clasificación de sonidos ambientales. Consta de grabaciones de 5 segundos de duración, organizadas en 50 clases (con 40 ejemplos por clase) y agrupadas en 5 categorías: animales, ruidos naturales, interior, exterior y actividades humanas. Complementa los datasets anteriores con sonidos del entorno cotidiano (pasos, lluvia, reloj, etc.), que añaden contexto al modelo.

### **2.4 AudioSet (HuggingFace)**
**Fuente:** https://huggingface.co/datasets/agkphysics/AudioSet  

Se trata de la versión completa de AudioSet, accesible a través de HuggingFace. Lo usamos para complementar las clases subrepresentadas en el dataset de Zenodo, especialmente las categorías de emergencia (`siren`, `gunshot`, `alarm`).

### **Resumen de fuentes**

| Dataset | Clips | Duración media | Clases | Responsable |
|---|---|---|---|---|
| UrbanSound8K (Kaggle) | 8.732 | 4s | 10 | Alessandra |
| FSD50K (Zenodo) | 51.197 | variable (de 0.3s a 30s) | filtradas | Andrés |
| ESC-50 (GitHub) | 2.000 | 5s | 50 | Natalia |
| AudioSet (HuggingFace) | complementario | 10s | filtradas | Natalia |
| **Total** | **≥ 60.000** | variable | | |

### **3. Estructura del proyecto**

```
proyecto4geeks/
│
├── data/
│   ├── raw/ --> Audios y CSVs originales sin modificar
│   │   ├── urbansound8k/
│   │   ├── zenodo/
│   │   ├── esc50/
│   │   └── audioset/
│   ├── interim/ --> Audios filtrados + CSVs homogeneizados por dataset
│   │   ├── urbansound8k/
│   │   ├── zenodo/
│   │   ├── esc50/
│   │   └── audioset/
│   └── processed/ --> Dataset final listo para tanto para el EDA como para el entrenamiento del modelo
│
├── src/data/build/
│   ├── dataset.py --> Clase Dataset y utilidades generales
│   ├── metadata.py (MetadataEX) --> Extracción de metadatos de audio
│   ├── urbansound8k/ --> Scripts específicos de UrbanSound8K
│   ├── zenodo_ds/ --> Scripts específicos de AudioSet/Zenodo
│   ├── esc50/ --> Scripts específicos de ESC-50
│   └── audioset/ --> Scripts específicos de AudioSet HuggingFace
│
├── notebooks/
│   ├── 00_building_ds.ipynb --> Este notebook
│   ├── 01_eda.ipynb
│   ├── 02_preprocessing.ipynb
│   ├── 03_modeling.ipynb
│   └── 04_evaluation.ipynb
│
├── deployment/ --> App web (Flask / Streamlit)
├── experiments/ --> Resultados de nuestros experimentos y métricas
└── src/ --> Código fuente del modelo
```

### **4. Proceso de construcción del dataset**

## 4. Pipeline de construcción del dataset

El proceso de construcción del dataset final sigue estos pasos para cada fuente:

```
[Descarga]  →  data/raw/<dataset>/
            ↓
[Filtrado de clases relevantes]  
            ↓
[Extracción de metadatos de audio]  
            ↓
[Generación de CSV homogeneizado]  
            ↓
[Unión de todos los CSVs]  
```

### **5. Descripción de las columnas del dataset final**
El dataset final contiene **20 columnas**, cumpliendo el requisito del proyecto. Incluye variables de identificación, de etiquetado, de seguridad y metadatos técnicos del audio.

### **5.1 Variables de identificación**

| Columna | Tipo | Ejemplo | Descripción |
|---|---|---|---|
| `columna` | str | `id_label, audio, human_label` | Cabecera del CSV. |
| `id_label` | int | `6` | Identificador de la clase. Es la **variable objetivo** del modelo de clasificación. |
| `audio` | str | `64760.wav` | Nombre del archivo de audio tal como viene en el dataset original. |
| `folder` | str | `surprised` | Carpeta donde está almacenado el audio. |
| `dataset_source` | str | `zenodo` | Dataset de origen del fragmento: `urbansound8k`, `zenodo`, `esc50` o `audioset`. Permite filtrar por fuente. |

### **5.2 Variables de etiquetado**

| Columna | Tipo | Ejemplo | Descripción |
|---|---|---|---|
| `label` | str | `/m/02sgy` | Identificador de clase en el vocabulario original del dataset fuente.|
| `human_label` | str | `guitar` | Clase principal homogeneizada en lenguaje natural. **Es la columna más importante para el modelo y la que se muestra en la aplicación web.** |
| `labels` | list[str] | `["/m/02sgy", "/m/0342h", "/m/0fx80y", "/m/04szw", "/m/04rlf"]` | Lista completa de todos los identificadores de clase del clip en el vocabulario original. Un clip puede tener múltiples sonidos simultáneos. |
| `human_labels` | list[str] | `["guitar", "strings", "Musical_instrument", "Music"]` | Lista completa de todas las clases homogeneizadas del clip. |

### **5.3 Variables de seguridad**

| Columna | Tipo | Valores | Descripción |
|---|---|---|---|
| `danger_level` | str | `2`, `0` | Nivel de peligro asignado a la clase. Es una **variable categórica** clave para el sistema de alertas. |
| `emergency` | bool | `True`, `False` | Variable derivada de `danger_level`. Permite formular el problema como una **clasificación binaria** (¿es o no es una emergencia?). |
| `environment` | str | `exterior`, `interior`, `indefinida` | Categoría de entorno sonoro inferida de la clase. Ejemplos: `traffic` --> `exterior`, `doors` --> `interior`, `guitar` --> `indefinida`. Es útil para contextualizar las alertas. |

### **5.4 Variables de partición**

| Columna | Tipo | Valores | Descripción |
|---|---|---|---|
| `split` | str | `train`, `test` | Conjunto de entrenamiento o prueba al que pertenece el fragmento. En UrbanSound8K, se respetan los 10 folds originales. En el resto, se aplica una división 80/20 por clase para evitar el desbalanceo. |

### **5.5 Metadatos técnicos del audio**

| Columna | Tipo | Ejemplo | Descripción |
|---|---|---|---|
| `duration` | float | `4.0` | Duración del fragmento en segundos. Varía entre datasets: UrbanSound8K 4s, AudioSet 10s.|
| `sample_rate` | int | `48000` | Frecuencia de muestreo en Hz. |
| `channels` | int | `Mono` | Número de canales: mono o estéreo.|
| `bit_depth` | int | `16` | Profundidad de bits de la señal (bits por muestra). Indica la resolución de amplitud. |
| `bit_velocity` | float | `1411 kbps` | Tasa de bits en kbps (kbits/segundo). Se calcula como `sample_rate × bit_depth × channels / 1000`. Indica la calidad y el peso del audio. |
| `size` | float | `16 bits` | Tamaño del archivo en kilobytes. Es útil para detectar clips muy pequeños o muy grandes. |
| `date_modification` | datetime | `04/02/2020 18:40` | Fecha de última modificación del fichero. |

### **Variables que NO usamos como features del modelo**
- **`audio`, `folder`**: son identificadores; no nos proporcionan información predictora.
- **`label`, `labels`**: son los IDs en su vocabulario original.
- **`date_modification`**: metadato de gestión; no tiene valor predictivo.
- **`human_labels`**: la lista completa se analiza en el EDA, pero no entra directamente al modelo.
- **`bit_velocity`**: se puede derivar de `sample_rate`, `bit_depth` y `channels`, por lo que es redundante como feature.

### **6. Constucción del dataset: código**

In [1]:
import os
import pandas as pd
from pathlib import Path
from src.data.build.metadata import MetadataEX
import src.data.build.zenodo_ds.zenodo_ds as zenodo
from src.data.build.dataset import Dataset

In [2]:
# Rutas del proyecto
def get_project_root():
    return Path().resolve()

root = get_project_root().parent
data_dir = root / "data"
raw_dir = data_dir / "raw"

csv_destiny_path = raw_dir / "zenodo.csv"
audio_folder = raw_dir / "zenodo"

data_build_dir = root / "src" / "data" / "build"
zenodo_dir = data_build_dir / "zenodo_ds"

csv_path = data_dir / "dev.csv"


In [3]:
df_original = pd.read_csv(os.path.join(zenodo_dir, "dev.csv"))
sinonimos = Dataset.cargar_sinonimos_csv(data_build_dir /"sinonimosV2.csv")

# Clases que quieres conservar (ahora puedes usar los nombres oficiales)
nombre_salida = os.path.join(raw_dir, "zenodo.csv")
df_res = zenodo.generar_csv_audio(
    df=df_original,
    col_labels='labels',
    col_mids='mids',
    col_fname='fname',
    ruta_carpeta="zenodo/",
    sinonimos=sinonimos,
    nombre_salida=nombre_salida,
    dataset_name="zenodo"
)
print(df_res.head())

[WARN] 'Rattle_(instrument)' duplicado → se mantiene en 'percussion', se ignora 'glass_metal'
[WARN] 'Sliding_door' duplicado → se mantiene en 'doors', se ignora 'movement'

Resumen: 2 conflictos detectados
Éxito. Archivo guardado en: /workspaces/proyecto4geeks/data/raw/zenodo.csv
       audio     label                                             labels  \
0  64760.wav  /m/02sgy  ["/m/02sgy", "/m/0342h", "/m/0fx80y", "/m/04sz...   
1  16399.wav  /m/02sgy  ["/m/02sgy", "/m/0342h", "/m/0fx80y", "/m/04sz...   
2  16401.wav  /m/02sgy  ["/m/02sgy", "/m/0342h", "/m/0fx80y", "/m/04sz...   
3  16402.wav  /m/02sgy  ["/m/02sgy", "/m/0342h", "/m/0fx80y", "/m/04sz...   
4  16404.wav  /m/02sgy  ["/m/02sgy", "/m/0342h", "/m/0fx80y", "/m/04sz...   

  human_label                                       human_labels  \
0      guitar  ["guitar", "strings", "Musical_instrument", "M...   
1      guitar  ["guitar", "strings", "Musical_instrument", "M...   
2      guitar  ["guitar", "strings", "Musical_instr

In [4]:
metadata = MetadataEX(
    csv_path=str(csv_destiny_path),
    dataset_name="zenodo",
    folder=str(audio_folder)
)

metadata.generate_metadata_audio()

Error: No se encontró la carpeta de audios en /workspaces/proyecto4geeks/data/raw/zenodo


### **7. Resumen del dataset final**
- **Total de instancias:** 60.000
- **Variables predictoras:** 20 columnas
- **Variables categóricas:** `human_label`, `danger_level`, `environment`, `dataset_source`, `split`
- **Variable objetivo principal:** `id_label` (clasificación multiclase)
- **Fuentes combinadas:** UrbanSound8K | Zenodo | ESC-50 | AudioSet